# Milestone 2 — Cleaning & Data Quality Mobile Legends
**Tujuan:** membersihkan CSV dengan Polars lazy, menyimpan Parquet, menjalankan DuckDB `SUMMARIZE` sebelum/sesudah, serta memvalidasi dan mencatat setiap pengurangan baris.

Notebook belum dieksekusi pada CSV asli. Jalankan seluruh cell berurutan untuk memperoleh angka nyata. Data mentah tidak diubah; output dengan nama sama diganti saat menjalankan ulang. Ruang disk tambahan diperlukan untuk staging, karantina, duplikat, Parquet final, dan spill DuckDB.

## Kebijakan cleaning
| Masalah | Keputusan dan alasan |
|---|---|
| Teks kosong / whitespace | Strip tepi; string kosong menjadi null. Isi ulasan, emoji, huruf, dan bahasa dipertahankan. |
| ID, content, score, at tidak valid | Karantina untuk analisis rating-temporal; simpan alasan, bukan hapus diam-diam. |
| ID duplikat | Setelah karantina, pertahankan baris valid pertama menurut posisi CSV. Simpan baris duplikat terpisah; tidak mengklaim baris pertama adalah versi terbaru. |
| Rating | Harus integer 1–5; tidak mengimputasi rating. |
| Thumbs up | Integer nonnegatif; nilai invalid menjadi null dan diberi flag, tidak dianggap nol. |
| Tanggal balasan invalid/sebelum ulasan | Jadikan null dan flag; ulasan tetap dipertahankan. |
| Kolom opsional hilang | Ditambahkan null dan dicatat; tidak menebak versi atau balasan. |
| Outlier | Tandai thumbs up dan panjang ulasan memakai IQR; pertahankan karena nilai ekstrem bisa sah. |

Tanggal tanpa zona dipertahankan sebagai waktu sumber. Tanggal berzona dikonversi ke UTC lalu disimpan tanpa zona. Karena zona asal waktu naive belum diketahui, durasi balasan adalah indikatif bila sumber mencampur zona. Tidak menebak negara pengguna dari nama/rating; relevansi Indonesia perlu didokumentasikan dari sumber/language coverage.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, platform, time
# Berlaku jika diset sebelum import Polars pertama setelah restart kernel.
os.environ['POLARS_MAX_THREADS'] = '2'
import polars as pl
import duckdb
from IPython.display import display

# Dapat dijalankan dari root project, notebooks/, Windows, atau Docker.
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
WINDOWS_CSV = Path(r'C:\Users\ASUS\Documents\Analisis BIGDATA\Tugas 1\tugas1-abd-2026\data\raw\archive\mobilelegends.csv')
DATA_PATH = Path(os.environ.get('M2_DATA_PATH', str(ROOT / 'data/raw/archive/mobilelegends.csv')))
if not DATA_PATH.is_file() and WINDOWS_CSV.is_file():
    DATA_PATH = WINDOWS_CSV
    ROOT = WINDOWS_CSV.parents[3]
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'CSV tidak ditemukan: {DATA_PATH}. Sesuaikan DATA_PATH pada cell ini.')
PROCESSED = ROOT / 'data/processed'
REPORTS = ROOT / 'output/milestone2'
for d in [PROCESSED, REPORTS, REPORTS / 'duckdb_tmp']:
    d.mkdir(parents=True, exist_ok=True)
STAGED = PROCESSED / 'mobilelegends_staged.parquet'
QUARANTINE = PROCESSED / 'mobilelegends_quarantine.parquet'
DEDUP = PROCESSED / 'mobilelegends_deduplicated.parquet'
DUPLICATES = PROCESSED / 'mobilelegends_duplicates.parquet'
CLEAN = PROCESSED / 'mobilelegends_clean.parquet'
STARTED = time.perf_counter()
print('CSV:', DATA_PATH)
print('Ukuran MB:', round(DATA_PATH.stat().st_size / 1_000_000, 2))
print('Polars:', pl.__version__, '| DuckDB:', duckdb.__version__)

CSV: /home/jovyan/work/data/raw/archive/mobilelegends.csv
Ukuran MB: 3088.87
Polars: 1.44.2 | DuckDB: 1.5.5


## 1. Scan CSV dan pemeriksaan schema
Semua kolom CSV dibaca sebagai string terlebih dahulu agar inference sampel tidak menyebabkan angka rusak terselip. Parsing CSV bersifat ketat: tidak memakai `ignore_errors`, tidak memotong baris bermasalah. Null literal seperti `NA`/`null` tidak dihapus karena bisa merupakan isi ulasan yang sah. Baris kosong diberi null oleh pembaca CSV.

In [2]:
raw = pl.scan_csv(DATA_PATH, infer_schema=False, encoding='utf8', row_index_name='_source_row')
names = raw.collect_schema().names()
required = ['reviewId', 'content', 'score', 'at']
missing_required = sorted(set(required) - set(names))
if missing_required:
    raise ValueError(f'Kolom wajib hilang: {missing_required}; sesuaikan mapping dengan header CSV asli.')
optional = ['userName', 'userImage', 'thumbsUpCount', 'reviewCreatedVersion', 'replyContent', 'repliedAt']
missing_optional = sorted(set(optional) - set(names))
if missing_optional:
    raw = raw.with_columns([pl.lit(None, dtype=pl.String).alias(c) for c in missing_optional])
print('Kolom opsional ditambahkan null:', missing_optional)
display(raw.head(5).collect(engine='streaming'))

Kolom opsional ditambahkan null: []


_source_row,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
u32,str,str,str,str,str,str,str,str,str,str,str
0,"""9132a8ef-cf00-4812-836a-8a6b86…","""Pengguna Google""","""https://play-lh.googleusercont…","""WOEE MONTOONN KASII GUA WS, CA…","""1""","""0""","""2.2.16.12322""","""2026-09-27 07:20:57""","""""","""""","""2.2.16.12322"""
1,"""55eccf21-3a36-49ff-89e7-73ff6a…","""Pengguna Google""","""https://play-lh.googleusercont…","""sangat bagus""","""5""","""0""","""2.2.16.12322""","""2026-09-27 07:20:27""","""""","""""","""2.2.16.12322"""
2,"""21e33d47-dd1f-49ba-8bd1-03aea8…","""Pengguna Google""","""https://play-lh.googleusercont…","""sangat baik""","""4""","""0""","""2.2.16.12322""","""2026-09-27 07:20:14""","""""","""""","""2.2.16.12322"""
3,"""57f89308-7935-435e-ae53-c2a932…","""Pengguna Google""","""https://play-lh.googleusercont…","""bagus sih tapi cuma itu selalu…","""4""","""0""","""2.2.16.12322""","""2026-09-27 07:18:11""","""""","""""","""2.2.16.12322"""
4,"""e4a2bed7-4f62-4825-8c74-64935f…","""Pengguna Google""","""https://play-lh.googleusercont…","""main solo history aja menang k…","""1""","""0""","""2.2.16.12322""","""2026-09-27 07:18:09""","""""","""""","""2.2.16.12322"""


## 2. Normalisasi, parsing, dan aturan validasi
Nilai asli numeric/tanggal disimpan di kolom `*_raw`. Posisi `_source_row` adalah index record CSV (mulai 0), bukan nomor baris fisik karena ulasan dapat memiliki newline.

In [3]:
def parse_date(column):
    x = pl.col(column)
    naive = [x.str.to_datetime(format=f, strict=False) for f in
             ['%Y-%m-%d %H:%M:%S%.f', '%Y-%m-%dT%H:%M:%S%.f', '%Y-%m-%d']]
    zoned = x.str.to_datetime(format='%Y-%m-%dT%H:%M:%S%.f%#z', strict=False,
                             time_zone='UTC').dt.replace_time_zone(None)
    return pl.coalesce([*naive, zoned])

strings = [c for c, t in raw.collect_schema().items() if t == pl.String]
normalized = raw.with_columns([
    pl.when(pl.col(c).str.strip_chars() == '').then(None)
      .otherwise(pl.col(c).str.strip_chars()).alias(c) for c in strings
])
backup = ['score', 'thumbsUpCount', 'at', 'repliedAt']
normalized = normalized.with_columns([pl.col(c).alias(c + '_raw') for c in backup])
# Nilai seperti '3.5' / '5.0' tidak dibulatkan secara diam-diam.
normalized = normalized.with_columns(
    pl.col('score').cast(pl.Int64, strict=False),
    pl.col('thumbsUpCount').cast(pl.Int64, strict=False),
    parse_date('at').alias('at'), parse_date('repliedAt').alias('repliedAt'),
    pl.col('content').str.len_chars().alias('content_length')
).with_columns(
    pl.col('reviewId').is_null().alias('invalid_review_id'),
    pl.col('content').is_null().alias('invalid_content'),
    (~pl.col('score').is_between(1, 5)).fill_null(True).alias('invalid_score'),
    pl.col('at').is_null().alias('invalid_review_date'),
    (pl.col('thumbsUpCount_raw').is_not_null() &
     (pl.col('thumbsUpCount').is_null() | (pl.col('thumbsUpCount') < 0))).fill_null(False).alias('invalid_thumbs'),
    (pl.col('repliedAt_raw').is_not_null() & pl.col('repliedAt').is_null()).alias('invalid_reply_date'),
    (pl.col('repliedAt') < pl.col('at')).fill_null(False).alias('reply_before_review')
).with_columns(
    pl.any_horizontal('invalid_review_id', 'invalid_content', 'invalid_score', 'invalid_review_date').alias('quarantined'),
    pl.concat_str([
        pl.when(pl.col(flag)).then(pl.lit(label)).otherwise(pl.lit(''))
        for flag, label in [('invalid_review_id', 'missing_id;'), ('invalid_content', 'missing_content;'),
                            ('invalid_score', 'invalid_score;'), ('invalid_review_date', 'invalid_at;')]
    ]).alias('quarantine_reason'),
    pl.when(pl.col('invalid_thumbs')).then(None).otherwise(pl.col('thumbsUpCount')).alias('thumbsUpCount'),
    pl.when(pl.col('invalid_reply_date') | pl.col('reply_before_review')).then(None)
      .otherwise(pl.col('repliedAt')).alias('repliedAt')
).with_columns(
    pl.col('replyContent').is_not_null().alias('has_reply'),
    ((pl.col('repliedAt') - pl.col('at')).dt.total_seconds() / 3600).alias('reply_delay_hours')
)
normalized.sink_parquet(STAGED, compression='zstd', engine='streaming')
print('Staging tersimpan:', STAGED)

Staging tersimpan: /home/jovyan/work/data/processed/mobilelegends_staged.parquet


## 3. DuckDB SUMMARIZE sebelum seleksi baris
`before` adalah staging yang sudah dinormalisasi dan diberi tipe/flag, tetapi belum dikarantina atau dideduplikasi. Nilai mentah tersedia di `*_raw`. Ringkasan missing/empty mentah dihitung terpisah melalui Polars.
DuckDB menggunakan batas memori 1 GB dan folder spill. Batas ini tidak membatasi memori Polars; deduplikasi exact tetap membutuhkan state dan dapat berat pada RAM rendah.

In [4]:
con = duckdb.connect()
con.execute("SET memory_limit='1GB'")
con.execute('SET threads=2')
con.execute("SET temp_directory='" + (REPORTS / 'duckdb_tmp').as_posix().replace("'", "''") + "'")
def view(name, path):
    escaped = path.as_posix().replace("'", "''")
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{escaped}')")
def query(sql):
    # Hanya mengambil hasil agregat/ringkasan kecil, bukan seluruh data.
    cur = con.execute(sql)
    return pl.DataFrame(cur.fetchall(), schema=[c[0] for c in cur.description], orient='row')
def nrows(name):
    return con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]
view('before', STAGED)
raw_rows = nrows('before')
if raw_rows == 0:
    raise ValueError('CSV tidak berisi record data.')
summary_before = query('SUMMARIZE before')
summary_before.write_csv(REPORTS / 'summarize_before.csv')
display(summary_before)
raw_missing = raw.select([pl.col(c).is_null().sum().alias(c) for c in strings]).collect(engine='streaming').row(0, named=True)
missing_normalized = query('SELECT ' + ', '.join(
    f'count(*) FILTER (WHERE "{c}" IS NULL) AS "{c}"' for c in required + optional) + ' FROM before')
missing_before = pl.DataFrame([{'column': c, 'raw_null_count': raw_missing.get(c),
    'normalized_null_count': missing_normalized[c][0],
    'normalized_null_percent': round(missing_normalized[c][0] / raw_rows * 100, 3)}
    for c in required + optional])
missing_before.write_csv(REPORTS / 'missing_before.csv')
display(missing_before)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""_source_row""","""UINTEGER""","""0""","""10448858""",9803105,"""5224429.0""","""3016325.9225249244""","""2614961""","""5222039""","""7836156""",10448859,0.00
"""reviewId""","""VARCHAR""","""0000041c-528b-4c18-a8a6-393e71…","""fffffda2-271b-488c-bf37-9c3f1a…",9995022,null,null,null,null,null,10448859,0.00
"""userName""","""VARCHAR""","""Pengguna Google""","""Pengguna Google""",1,null,null,null,null,null,10448859,0.00
"""userImage""","""VARCHAR""","""https://play-lh.googleusercont…","""https://play-lh.googleusercont…",1,null,null,null,null,null,10448859,0.00
"""content""","""VARCHAR""","""! !""","""󠀺󠀸󠀸󠀰󠀴󠀶󠀱󠀱󠀱󠀡󠀡󠀢󠀣󠀦󠀧󠀧󠀤󠀤󠀣󠀣󠀨󠀨󠀩󠀪󠀫󠀬󠀮󠀮󠀯󠀯…",6900143,null,null,null,null,null,10448859,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""reply_before_review""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,10448859,0.00
"""quarantined""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,10448859,0.00
"""quarantine_reason""","""VARCHAR""","""""","""missing_content;""",3,null,null,null,null,null,10448859,0.00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column,raw_null_count,normalized_null_count,normalized_null_percent
str,i64,i64,f64
"""reviewId""",0,0,0.0
"""content""",0,18,0.0
"""score""",0,0,0.0
"""at""",0,0,0.0
"""userName""",0,0,0.0
"""userImage""",0,0,0.0
"""thumbsUpCount""",0,0,0.0
"""reviewCreatedVersion""",0,3578132,34.244
"""replyContent""",0,10310120,98.672


## 4. Karantina dan deduplikasi Polars per partisi disk
Versi perbaikan: tidak lagi membuat state deduplikasi untuk seluruh ID sekaligus. DuckDB membagi baris valid ke 128 partisi Parquet menggunakan hash ID, dengan batas memori 1 GB dan spill disk. ID sama selalu berada dalam partisi sama. Polars melakukan deduplikasi dan anti-join audit hanya pada satu partisi setiap kali.

Setiap partisi diurutkan berdasarkan `_source_row` sebelum `keep='first'`, sehingga kebijakan tetap memilih baris valid pertama dalam CSV asli, meskipun penulisan partisi mengubah urutan record. Hasil partisi digabung melalui streaming. Ini mengurangi working set, tetapi tidak menjamin batas RAM keras bagi Polars. Bila satu partisi masih terlalu besar, naikkan `N_BUCKETS` menjadi 256 atau 512.

Folder sementara unik untuk tiap eksekusi; file staging asli tetap dipertahankan. Jika proses mati, folder `m2_partition_*` mungkin tertinggal di data/processed dan boleh dihapus setelah kernel berhenti. Jangan hapus CSV mentah.

In [5]:
import tempfile
N_BUCKETS = 128
stage = pl.scan_parquet(STAGED)
print('Menulis karantina...', flush=True)
stage.filter(pl.col('quarantined')).sink_parquet(QUARANTINE, compression='zstd', engine='streaming')

# Pembagian/penyimpanan disk memakai DuckDB; cleaning dan deduplikasi memakai Polars.
with tempfile.TemporaryDirectory(prefix='m2_partition_', dir=PROCESSED) as task_dir:
    task_dir = Path(task_dir)
    partition_dir = task_dir / 'valid'
    kept_dir = task_dir / 'kept'
    dup_dir = task_dir / 'duplicates'
    kept_dir.mkdir(); dup_dir.mkdir()
    safe_dir = partition_dir.as_posix().replace("'", "''")
    print(f'Membagi data valid ke {N_BUCKETS} partisi disk...', flush=True)
    con.execute(f"""
        COPY (SELECT *, hash(reviewId) % {N_BUCKETS} AS _bucket
              FROM before WHERE NOT quarantined)
        TO '{safe_dir}' (FORMAT PARQUET, PARTITION_BY (_bucket), COMPRESSION ZSTD)
    """)
    bucket_paths = sorted(partition_dir.glob('_bucket=*'))
    if not bucket_paths:
        raise ValueError('Tidak ada record valid. Periksa karantina sebelum melanjutkan.')
    for i, bucket in enumerate(bucket_paths, 1):
        # File Parquet tidak menyertakan _bucket; hive parsing dinonaktifkan.
        part = pl.scan_parquet(str(bucket / '*.parquet'), hive_partitioning=False)
        kept_file = kept_dir / f'part_{i:04d}.parquet'
        dup_file = dup_dir / f'part_{i:04d}.parquet'
        part.sort('_source_row').unique(subset=['reviewId'], keep='first', maintain_order=False).sink_parquet(
            kept_file, compression='zstd', engine='streaming')
        ids = pl.scan_parquet(kept_file).select('_source_row')
        part.join(ids, on='_source_row', how='anti').sink_parquet(
            dup_file, compression='zstd', engine='streaming')
        if i == 1 or i % 16 == 0 or i == len(bucket_paths):
            print(f'Partisi {i}/{len(bucket_paths)} selesai', flush=True)
    print('Menggabungkan hasil secara streaming...', flush=True)
    pl.scan_parquet(str(kept_dir / '*.parquet')).sink_parquet(DEDUP, compression='zstd', engine='streaming')
    pl.scan_parquet(str(dup_dir / '*.parquet')).sink_parquet(DUPLICATES, compression='zstd', engine='streaming')

view('quarantine', QUARANTINE)
view('dedup', DEDUP)
view('duplicates', DUPLICATES)
quarantined_rows, duplicate_rows, dedup_rows = nrows('quarantine'), nrows('duplicates'), nrows('dedup')
assert raw_rows == quarantined_rows + duplicate_rows + dedup_rows
print(f'Mentah={raw_rows:,}; karantina={quarantined_rows:,}; duplikat={duplicate_rows:,}; valid unik={dedup_rows:,}')
if dedup_rows == 0:
    raise ValueError('Tidak ada record valid. Periksa quarantine sebelum melanjutkan.')

Menulis karantina...
Membagi data valid ke 128 partisi disk...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Partisi 1/128 selesai
Partisi 16/128 selesai
Partisi 32/128 selesai
Partisi 48/128 selesai
Partisi 64/128 selesai
Partisi 80/128 selesai
Partisi 96/128 selesai
Partisi 112/128 selesai
Partisi 128/128 selesai
Menggabungkan hasil secara streaming...
Mentah=10,448,859; karantina=24; duplikat=0; valid unik=10,448,835


## 5. Outlier IQR — ditandai, dipertahankan
Ambang dihitung sesudah karantina dan deduplikasi menggunakan exact `quantile_cont`. Jika IQR nol, setiap nilai di luar batas tetap ditandai, tetapi interpretasinya perlu hati-hati. Tidak ada winsorization atau penghapusan rating rendah/tinggi. Null numeric tetap null dan flag outlier bernilai false.

In [6]:
outlier_rules = {}
for c in ['thumbsUpCount', 'content_length']:
    q1, q3 = con.execute(f'SELECT quantile_cont("{c}", 0.25), quantile_cont("{c}", 0.75) FROM dedup').fetchone()
    outlier_rules[c] = {'q1': q1, 'q3': q3,
        'lower': None if q1 is None else q1 - 1.5 * (q3 - q1),
        'upper': None if q3 is None else q3 + 1.5 * (q3 - q1)}
flags = []
for c, bounds in outlier_rules.items():
    expression = pl.lit(False) if bounds['lower'] is None else (
        (pl.col(c) < bounds['lower']) | (pl.col(c) > bounds['upper'])).fill_null(False)
    flags.append(expression.alias(c + '_outlier'))
pl.scan_parquet(DEDUP).with_columns(flags).sink_parquet(CLEAN, compression='zstd', engine='streaming')
view('clean', CLEAN)
display(pl.DataFrame([{'column': c, **v} for c, v in outlier_rules.items()]))

column,q1,q3,lower,upper
str,f64,f64,f64,f64
"""thumbsUpCount""",0.0,0.0,0.0,0.0
"""content_length""",9.0,63.0,-72.0,144.0


## 6. SUMMARIZE sesudah cleaning dan validasi akhir
Setiap aturan diuji pada seluruh Parquet final. Assertions gagal jika ada pelanggaran atau perhitungan jumlah baris tidak konsisten.

In [7]:
summary_after = query('SUMMARIZE clean')
summary_after.write_csv(REPORTS / 'summarize_after.csv')
display(summary_after)
checks = query("""
SELECT count(*) AS clean_rows,
 count(*) - count(DISTINCT reviewId) AS duplicate_ids,
 count(*) FILTER (WHERE reviewId IS NULL OR trim(reviewId)='') AS missing_ids,
 count(*) FILTER (WHERE content IS NULL OR trim(content)='') AS empty_content,
 count(*) FILTER (WHERE score IS NULL OR score NOT BETWEEN 1 AND 5) AS bad_scores,
 count(*) FILTER (WHERE "at" IS NULL) AS bad_dates,
 count(*) FILTER (WHERE thumbsUpCount < 0) AS negative_thumbs,
 count(*) FILTER (WHERE repliedAt < "at") AS negative_reply_time,
 count(*) FILTER (WHERE quarantined) AS quarantine_leak,
 count(*) FILTER (WHERE thumbsUpCount_outlier) AS thumbs_outliers,
 count(*) FILTER (WHERE content_length_outlier) AS length_outliers
FROM clean
""").row(0, named=True)
assert checks['clean_rows'] == dedup_rows
for c in ['duplicate_ids', 'missing_ids', 'empty_content', 'bad_scores', 'bad_dates',
          'negative_thumbs', 'negative_reply_time', 'quarantine_leak']:
    assert checks[c] == 0, f'Validasi gagal: {c}={checks[c]}'
assert raw_rows == quarantined_rows + duplicate_rows + checks['clean_rows']
pl.DataFrame([checks]).write_csv(REPORTS / 'validation.csv')
flags_report = query('SELECT ' + ', '.join(f'count(*) FILTER (WHERE {c}) AS {c}' for c in
    ['invalid_review_id', 'invalid_content', 'invalid_score', 'invalid_review_date',
     'invalid_thumbs', 'invalid_reply_date', 'reply_before_review']) + ' FROM before')
flags_report.write_csv(REPORTS / 'quality_flags.csv')
missing_after = query('SELECT ' + ', '.join(
    f'count(*) FILTER (WHERE "{c}" IS NULL) AS "{c}"' for c in required + optional) + ' FROM clean')
pl.DataFrame([{'column': c, 'null_count': missing_after[c][0],
              'null_percent': round(missing_after[c][0] / checks['clean_rows'] * 100, 3)}
              for c in required + optional]).write_csv(REPORTS / 'missing_after.csv')
display(pl.DataFrame([checks]))
display(flags_report)
print('Semua validasi akhir lulus.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""_source_row""","""UINTEGER""","""0""","""10448858""",9803105,"""5224426.4289507875""","""3016323.3747331635""","""2613271""","""5224739""","""7839816""",10448835,0.00
"""reviewId""","""VARCHAR""","""0000041c-528b-4c18-a8a6-393e71…","""fffffda2-271b-488c-bf37-9c3f1a…",9995022,null,null,null,null,null,10448835,0.00
"""userName""","""VARCHAR""","""Pengguna Google""","""Pengguna Google""",1,null,null,null,null,null,10448835,0.00
"""userImage""","""VARCHAR""","""https://play-lh.googleusercont…","""https://play-lh.googleusercont…",1,null,null,null,null,null,10448835,0.00
"""content""","""VARCHAR""","""! !""","""󠀺󠀸󠀸󠀰󠀴󠀶󠀱󠀱󠀱󠀡󠀡󠀢󠀣󠀦󠀧󠀧󠀤󠀤󠀣󠀣󠀨󠀨󠀩󠀪󠀫󠀬󠀮󠀮󠀯󠀯…",6900143,null,null,null,null,null,10448835,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""quarantine_reason""","""VARCHAR""","""""","""""",1,null,null,null,null,null,10448835,0.00
"""has_reply""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,10448835,0.00
"""reply_delay_hours""","""DOUBLE""","""0.0038888888888888888""","""4835.663333333333""",57484,"""18.275236359473425""","""40.899461706471136""","""2.068975445712411""","""6.867116245653267""","""18.184196031675917""",10448835,98.88


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

clean_rows,duplicate_ids,missing_ids,empty_content,bad_scores,bad_dates,negative_thumbs,negative_reply_time,quarantine_leak,thumbs_outliers,length_outliers
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
10448835,0,0,0,0,0,0,0,0,410901,894876


invalid_review_id,invalid_content,invalid_score,invalid_review_date,invalid_thumbs,invalid_reply_date,reply_before_review
i64,i64,i64,i64,i64,i64,i64
0,18,6,0,0,0,22030


Semua validasi akhir lulus.


## 7. Audit dan kesimpulan otomatis
Checksum dibaca bertahap. Ringkasan mencatat versi dependency, aturan, waktu, ukuran asli, dan jumlah baris sebenarnya. Baris flag dapat saling tumpang tindih, sehingga jumlah flag tidak sama dengan jumlah karantina. `dataset_meets_size_rule` hanya memeriksa skala, tidak membuktikan relevansi Indonesia atau lisensi.

In [8]:
hash_value = hashlib.sha256()
with DATA_PATH.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        hash_value.update(block)
file_bytes = DATA_PATH.stat().st_size
meets_size = file_bytes >= 500_000_000 or raw_rows > 1_000_000
report = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_name': DATA_PATH.name, 'input_bytes': file_bytes, 'sha256': hash_value.hexdigest(),
    'python': platform.python_version(), 'polars': pl.__version__, 'duckdb': duckdb.__version__,
    'missing_optional_columns': missing_optional,
    'raw_rows': raw_rows, 'quarantined_rows': quarantined_rows,
    'duplicate_rows_removed': duplicate_rows, 'clean_rows': checks['clean_rows'],
    'dataset_meets_size_rule': meets_size, 'outlier_rules': outlier_rules,
    'validation': checks, 'quality_flags': flags_report.row(0, named=True),
    'elapsed_seconds': round(time.perf_counter() - STARTED, 2),
    'dedup_partitions': N_BUCKETS,
    'dedup_policy': 'first valid record in source CSV order',
    'outlier_policy': 'retain and flag; IQR 1.5 after deduplication',
    'date_policy': 'naive source times retained; offset times normalized to UTC naive'
}
(REPORTS / 'cleaning_report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
conclusion = f'''# Hasil Milestone 2

Dihasilkan otomatis pada {report['generated_at_utc']} setelah validasi akhir lulus.

- Ukuran CSV asli: {file_bytes / 1_000_000:,.2f} MB; {raw_rows:,} record.
- Memenuhi batas skala tugas (>=500 MB atau >1 juta baris): {meets_size}.
- Karantina karena ID/content/rating/tanggal ulasan invalid: {quarantined_rows:,} record.
- Duplikat ID valid yang dipisahkan: {duplicate_rows:,} record.
- Dataset final: {checks['clean_rows']:,} ulasan valid dengan ID unik.
- Outlier thumbs up: {checks['thumbs_outliers']:,}; panjang ulasan: {checks['length_outliers']:,}. Semuanya dipertahankan.
- SUMMARIZE sebelum seleksi dan sesudah cleaning tersedia di CSV laporan.
- Detail null/flag/ambang tersedia di missing_*.csv, quality_flags.csv, dan cleaning_report.json.

## Interpretasi yang perlu diperiksa mahasiswa

Karantina dapat mengubah komposisi rating/periode: bandingkan distribusi sebelum/sesudah sebelum Milestone 3.
Null pada replyContent berarti balasan tidak tersedia dalam snapshot; tidak membuktikan pengembang tidak pernah membalas.
Missing version/thumbs up tidak diimputasi. Audit konflik ID pada file duplicates sebelum menafsirkan hasil.
Timezone dan relevansi Indonesia belum otomatis diverifikasi oleh pipeline ini.

## AI Disclosure Statement

Alat: ChatGPT/Codex. Bantuan: rancangan pipeline Polars, profiling DuckDB, validasi, dan dokumentasi Milestone 2.
Status teknis: seluruh cell pada eksekusi ini selesai dan assertions lulus.
Pemahaman dan verifikasi mahasiswa: isi sendiri setelah memeriksa output, meninjau keputusan cleaning, dan memahami kode.
'''
(REPORTS / 'milestone2_results.md').write_text(conclusion, encoding='utf-8')
print(conclusion)
con.close()

# Hasil Milestone 2

Dihasilkan otomatis pada 2026-10-09T16:13:10.151098+00:00 setelah validasi akhir lulus.

- Ukuran CSV asli: 3,088.87 MB; 10,448,859 record.
- Memenuhi batas skala tugas (>=500 MB atau >1 juta baris): True.
- Karantina karena ID/content/rating/tanggal ulasan invalid: 24 record.
- Duplikat ID valid yang dipisahkan: 0 record.
- Dataset final: 10,448,835 ulasan valid dengan ID unik.
- Outlier thumbs up: 410,901; panjang ulasan: 894,876. Semuanya dipertahankan.
- SUMMARIZE sebelum seleksi dan sesudah cleaning tersedia di CSV laporan.
- Detail null/flag/ambang tersedia di missing_*.csv, quality_flags.csv, dan cleaning_report.json.

## Interpretasi yang perlu diperiksa mahasiswa

Karantina dapat mengubah komposisi rating/periode: bandingkan distribusi sebelum/sesudah sebelum Milestone 3.
Null pada replyContent berarti balasan tidak tersedia dalam snapshot; tidak membuktikan pengembang tidak pernah membalas.
Missing version/thumbs up tidak diimputasi. Audit konflik ID pada

## 8. Setelah selesai
1. Periksa karantina/duplikat dengan `pl.scan_parquet(...).head(10).collect()`; jangan memuat file penuh hanya untuk preview.
2. Periksa laporan otomatis dan jelaskan alasan kebijakan cleaning dengan kata-kata sendiri.
3. Simpan notebook dengan output dari dataset asli. Jangan commit CSV/Parquet, karantina, duplikat, database, atau folder spill.
4. Update ukuran/jumlah baris/periode pada `data/README.md` dari hasil asli dan lengkapi disclosure penggunaan AI.
5. Commit Milestone 2 setelah eksekusi sukses. Milestone 3 memakai `data/processed/mobilelegends_clean.parquet`.

Referensi resmi: [Polars sink_parquet](https://docs.pola.rs/api/python/stable/reference/api/polars.LazyFrame.sink_parquet.html), [Polars unique](https://docs.pola.rs/api/python/stable/reference/api/polars.LazyFrame.unique.html), [DuckDB SUMMARIZE](https://duckdb.org/docs/stable/sql/statements/summarize).